# Project Work Log

## 2026-05-10

### Session: Code review, verification, and run

---

#### [20:55] Project exploration
- Explored repository structure and understood overall purpose
- Project: MSc AI Capstone — AI-Driven QoS Prediction in SD-WAN Networks (University of Galway)
- Goal: Supervised regression to predict `recommended_bandwidth_percent` from network flow features
- Four scripts identified: `generate_dataset.py`, `process_public_dataset.py`, `clean_public_dataset.py`, `create_cicids_project_aligned_sample.py`

---

#### [21:00] Code review — findings per script

**`src/generate_dataset.py`**
- Generates 50,000 synthetic SD-WAN flow records with reproducible seed (42)
- Rule-based target `recommended_bandwidth_percent` produces only 13 discrete integer values (35–95 in 5-step increments) — effectively a classification problem, not regression
- Features are sampled independently with no realistic correlations between them
- **Bug**: output path hardcoded to `data/synthetic/sdwan_qos_synthetic.csv` but README documents `data/sdwan_qos_synthetic.csv` — mismatch
- No CLI arguments (unlike the other three scripts)

**`src/process_public_dataset.py`**
- Chunked multi-format scanner (CSV, TSV, JSON, Parquet) for datasets under `data/raw/`
- Produces per-dataset Excel + CSV reports: row count, feature count, label distribution, missing/infinite value counts
- Well-structured with encoding fallback and per-dataset grouping
- No variance or correlation stats — useful addition for EDA before modelling

**`src/clean_public_dataset.py`**
- Two-pass design: profile first, then write cleaned output
- Removes constant-zero numeric columns, replaces infinities with NaN, drops rows with missing values
- Reports label imbalance with configurable threshold (default 60%)
- Only removes constant-zero columns — no near-constant, near-duplicate, or high-correlation filtering

**`src/create_cicids_project_aligned_sample.py`**
- Maps CICIDS2017 flow columns to project-style SD-WAN features using proxy heuristics
- `link_type = "unknown"` and `time_of_day = -1` for all rows — zero variance, useless for modelling
- `packet_loss_percent` is binary (0.0 or 1.0 based on RST flag), not a real percentage — misleading name
- `latency_ms` derived from Flow IAT Mean (inter-arrival time) — mean=1440ms, max=64100ms, far outside real network RTT range
- `application_type` heavily skewed: 97% of rows are `web` or `dns`; only 1 `video` row in 5,000
- `calculate_bandwidth` rule differs from `generate_dataset.py` (adds attack penalty, extra app types) — the two datasets produce incompatible target scales

---

#### [21:10] Code verification and live run

- Syntax-checked all four scripts: `python -m py_compile` — all passed
- **`generate_dataset.py`**: ran successfully, produced `data/synthetic/sdwan_qos_synthetic.csv` (50,000 rows, 12 columns, 0 missing values)
- **`process_public_dataset.py`**: ran successfully, scanned all 8 CICIDS2017 CSVs (2,830,743 rows, 79 features, 15 label classes), produced Excel report
- **`clean_public_dataset.py`**: ran successfully, cleaned CICIDS dataset to 2,827,876 rows, removed 8 constant-zero features, flagged as imbalanced (BENIGN = 80.3%)
- **`create_cicids_project_aligned_sample.py`**: ran successfully with `--sample-size 5000` test run, confirmed all data quality issues above from live output

---

#### [21:20] Confirmed bugs and recommended fixes (not yet applied)

| # | File | Issue | Priority |
|---|---|---|---|
| 1 | `generate_dataset.py:69` | Output path `data/synthetic/sdwan_qos_synthetic.csv` doesn't match README (`data/sdwan_qos_synthetic.csv`) | Low |
| 2 | `create_cicids_project_aligned_sample.py` | `time_of_day=-1` and `link_type="unknown"` are constant — drop before modelling | High |
| 3 | `create_cicids_project_aligned_sample.py` | `packet_loss_percent` is a binary RST flag, not a real percentage — rename | Medium |
| 4 | `create_cicids_project_aligned_sample.py` | `latency_ms` derived from IAT, not RTT — document clearly in thesis | Medium |
| 5 | Both scripts | `calculate_bandwidth` rule differs between synthetic and CICIDS scripts — align before cross-dataset comparison | High |

---

#### [21:25] Next steps discussed

- **Recommendation**: train baselines on synthetic data now, don't wait for new datasets
- Synthetic pipeline validates train/test split, encoding, model fitting, and evaluation end-to-end
- Key thesis limitation to address: both datasets use the same hand-written rule as ground truth — no real-world QoS label exists anywhere in the pipeline
- Planned models: Linear Regression, Random Forest, Gradient Boosting / XGBoost, SVR
- Planned metrics: MAE, RMSE, R², Inference Time
- Fix the 5 bugs above before writing the modelling notebook

---

## 2026-05-11

### Session: Generic cleaning fixes and verification

---

#### [08:58] Local work log setup
- Confirmed `WORKLOG.md` is the local daily task log for this project
- Confirmed `WORKLOG.md` is ignored by Git so it stays local
- Added this daily summary section for tracking work from now onwards

---

#### [09:00] Cleaning script improvements
- Fixed label cleaning so missing or blank labels are not counted as real classes
- Added configurable missing-value handling to `clean_public_dataset.py`
- Supported missing policies: `all`, `label`, `required`, and `none`
- Kept `--keep-missing-rows` as a shortcut for `--missing-policy none`
- Added validation so `--missing-policy required` must include `--required-columns`
- Added report output for missing required columns so column-name mistakes are visible

---

#### [09:10] Code review and verification
- Syntax-checked all scripts successfully
- Regenerated synthetic data successfully
- Ran the generic public dataset inspector successfully on CICIDS2017
- Ran the public dataset cleaner successfully on CICIDS2017
- Created a CICIDS project-aligned verification sample successfully
- Verified missing-policy behavior using synthetic data and temporary output folders

---

#### [09:15] Dataset status confirmed
- Synthetic dataset contains 50,000 rows plus header
- CICIDS raw inspection found 2,830,743 rows across 8 files
- CICIDS cleaned output contains 2,827,876 rows plus header
- Cleaner removed 8 constant-zero features
- CICIDS label distribution remains highly imbalanced, with BENIGN as the majority class

---


#### [09:35] Document alignment review
- Read project proposal, capstone guidelines, intro thesis slides, and work-log notebook
- Verified current code against stated research aim, datasets, feature engineering, preprocessing, and evaluation expectations
- Confirmed scripts compile successfully
- Identified main remaining gap: baseline model training and experimental evaluation are not implemented yet


#### [10:48] Baseline training script
- Added  for first modelling experiments
- Implemented DummyRegressor mean baseline and Linear Regression
- Added preprocessing for numeric and categorical project features
- Ran training on synthetic SD-WAN data and saved results to 


#### [11:04] Model results location
- Changed baseline training output from data/processed to reports/model_results/model_results.csv
- Updated README to document the model-results folder
- Reran baseline training and confirmed results are written to the new reports location


---

## 2026-05-13

### Session: Zenodo-first dataset pivot and leakage-safe modelling setup

---

#### [16:30] Supervisor feedback incorporated
- Parked BNN-UPC temporarily because the local folder currently contains documentation/assets only, not the uncompressed dataset and `datanetAPI` module
- Re-focused the first public-data workflow on the Zenodo 5G campus network dataset under `data/raw/Zenodo_13754300`
- Clarified that CICIDS2017 should be treated as exploratory/proxy data because it is an intrusion-detection dataset, not a QoS prediction dataset

---

#### [16:40] Zenodo exploration
- Updated `src/initial_data_set_exploration.py` to focus on `data/raw/Zenodo_13754300`
- Confirmed the script safely reads only the first 5 rows and counts rows without loading the large packet CSVs into memory
- Explored six Zenodo CSVs: four packet-level OWD files and two throughput files
- Identified the two throughput files as the first modelling source: `ntnu_tput_all_Throughput.csv` and `wue_tput_all_Throughput.csv`
- Identified useful inputs: `testbed`, `gnb`, `sdr`, `bw_mhz`, `slots`, `ratio`, `direction`, and `offered_throughput_mbps`
- Identified first target: `actual_throughput_mbps`

---

#### [16:50] Zenodo processing and leakage review
- Reviewed and verified `src/process_zenodo_dataset.py`
- Confirmed `--skip-owd` quick mode loads the two throughput files and creates 604 rows after uplink/downlink unpivoting
- Added `offered_throughput_mbps` to the processed output as the key controllable offered-load input
- Added code comments explaining that `recommended_bandwidth_percent` is derived from actual throughput divided by offered throughput
- Documented that training on `recommended_bandwidth_percent` while keeping `actual_throughput_mbps` as an input causes target leakage
- Added a runtime modelling note recommending `actual_throughput_mbps` as the first Zenodo prediction target

---

#### [16:55] Verification
- Syntax-checked `src/process_zenodo_dataset.py`, `src/train_baseline.py`, and `src/initial_data_set_exploration.py`
- Ran Zenodo processing with `--skip-owd` to temporary output successfully
- Confirmed processed quick output shape: 604 rows and 22 columns
- Ran baseline training against `actual_throughput_mbps` with leakage-prone derived/outcome columns dropped
- Fixed `src/train_baseline.py` so result reports show the selected `--target-column` instead of always showing `recommended_bandwidth_percent`
- Updated README with Zenodo dataset workflow, recommended target, leakage warning, and example baseline command


## 2026-05-13

### Session: Thesis action plan noted

---

#### [09:00] Action plan added to project context
- Located and read documents/Thesis_Action_Plan.docx
- Noted that future work should follow the action plan as the project guide
- Confirmed WORKLOG.ipynb should be updated for every session


---

### Session: Zenodo baseline wrapper, target options, and code comments

---

#### [19:40] Zenodo baseline wrapper added
- Added `src/train_zenodo_baseline.py` as a shorter wrapper around Zenodo processing and generic baseline training
- Default target is `actual_throughput_mbps`
- Added `--target` / `--target-column` choice for `actual_throughput_mbps` or `recommended_bandwidth_percent`
- Added separate default output for recommended-bandwidth runs: `reports/model_results/zenodo_baseline_recommended_bandwidth_results.csv`

---

#### [19:48] Target leakage handling verified
- Confirmed the model input columns exclude the selected target automatically
- Confirmed `actual_throughput_mbps` is dropped when predicting `recommended_bandwidth_percent`
- Confirmed `recommended_bandwidth_percent`, `jitter_ms`, `packet_loss_percent`, and `source_file` are dropped when predicting `actual_throughput_mbps`
- Verified both Zenodo target modes run successfully against temporary outputs

---

#### [20:00] Code comments and README updates
- Added explanatory comments across the project scripts to document data flow, feature engineering, cleaning decisions, leakage prevention, and baseline modelling steps
- Updated README with the Zenodo wrapper command and examples for both supported targets
- Syntax-checked all Python scripts with `.venv/bin/python -m py_compile src/*.py`

---

#### [20:05] Commit
- Committed verified code changes: `511b5ca Add Zenodo baseline target options and comments`


## 2026-05-14

### Session: Thesis document verification and fixes

---

#### [13:20] Thesis folder reviewed
- Verified the added thesis document folder: `MSc AI-DA-AI Online Thesis Document/`
- Checked the LaTeX source, bibliography, class files, figures, and generated PDF
- Identified stale template content, outdated supervisor/date details, missing local LaTeX package references, and methodology text that no longer matched the Zenodo-first QoS prediction direction

---

#### [13:45] Thesis content aligned with mentor feedback
- Updated supervisor details to Dr. Jawad Manzoor
- Reworked methodology wording to focus on QoS prediction from public datasets rather than relying on unavailable workplace SD-WAN data
- Prioritised the Zenodo 5G campus dataset as the first public dataset for modelling
- Kept CICIDS2017 positioned only as an exploratory/proxy dataset because it is primarily intrusion-detection traffic
- Noted BNN-UPC and MAWI as further public datasets to assess
- Added the 2025 MTEAL SD-WAN QoS/traffic-engineering paper suggested by Dr. Jawad to the literature review and bibliography

---

#### [14:05] LaTeX build fixed and verified
- Updated local thesis class package references so bundled style files load correctly from the `Classes/` folder
- Disabled optional template packages that were not needed for the current thesis draft and were unavailable in the local TinyTeX setup
- Added a small local `grfext.sty` compatibility shim for the PDF build environment
- Rebuilt `Thesis_template.pdf` successfully with `latexmk -pdf -interaction=nonstopmode -halt-on-error Thesis_template.tex`
- Verified the output PDF builds to 39 pages with no unresolved citation/reference errors
